In [1]:
from __future__ import annotations

import subprocess
import time
from datetime import datetime, timezone
from pathlib import Path
from concurrent.futures import (
    ThreadPoolExecutor,
    as_completed,
)
import pandas as pd

In [2]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent


SMOKE_MANIFEST = (
    PROJECT_ROOT
    / "data"
    / "samples"
    / "benchmark_sample.csv"
)

MINERU_EXE = (
    PROJECT_ROOT
    / ".venv-mineru"
    / "Scripts"
    / "mineru.exe"
)

OUTPUT_ROOT = (
    PROJECT_ROOT
    / "artifacts"
    / "mineru"
    / "smoke"
)

RUNS_PATH = OUTPUT_ROOT / "runs.parquet"
RUNS_CSV_PATH = OUTPUT_ROOT / "runs.csv"



OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f"MinerU: {MINERU_EXE}")
print(f"Manifesto: {SMOKE_MANIFEST}")
print(f"Saída: {OUTPUT_ROOT}")

MinerU: D:\baseia_v3\.venv-mineru\Scripts\mineru.exe
Manifesto: D:\baseia_v3\data\samples\benchmark_sample.csv
Saída: D:\baseia_v3\artifacts\mineru\smoke


In [3]:
BACKEND = "pipeline"

API_URLS = (
    "https://yl2qubpbnye7sd-8000.proxy.runpod.net",
    "https://m05ohopxxf0mkl-8000.proxy.runpod.net",
    "https://r4srxz2kdyhnx9-8000.proxy.runpod.net"
)


OVERWRITE = True

MAX_WORKERS = 24

TEST_SLICE = 24

In [4]:
if not MINERU_EXE.exists():
    raise FileNotFoundError(
        f"Executável do MinerU não encontrado: {MINERU_EXE}"
    )

if not SMOKE_MANIFEST.exists():
    raise FileNotFoundError(
        f"Manifesto não encontrado: {SMOKE_MANIFEST}"
    )


smoke_sample = pd.read_csv(SMOKE_MANIFEST)[:TEST_SLICE]

print(f"Documentos no smoke test: {len(smoke_sample)}")

smoke_sample[
    [
        "filename",
        "page_count",
        "size_mb",
        "path",
    ]
]

Documentos no smoke test: 24


,filename,page_count,size_mb,path
0,metadata_williamson_1999_bureaucracies.pdf,1.0,0.0025,D:\baseia_v3\corpus\metadata_williamson_1999_b...
1,Risk-Sharing Contracts and risk management of ...,1.0,0.9060,D:\baseia_v3\corpus\Risk-Sharing Contracts and...
2,New approach to financial time series forecast...,4.0,0.2299,D:\baseia_v3\corpus\New approach to financial ...
3,uerj_rede_sirius_procedimentos_bdtd.pdf,3.0,0.0423,D:\baseia_v3\corpus\uerj_rede_sirius_procedime...
4,GTL_0081.pdf,5.0,0.9256,D:\baseia_v3\corpus\GTL_0081.pdf
5,GTL_1171.pdf,5.0,0.8370,D:\baseia_v3\corpus\GTL_1171.pdf
6,GAE_0970.pdf,5.0,1.1228,D:\baseia_v3\corpus\GAE_0970.pdf
7,10.1080_096031096334006.pdf,8.0,0.1565,D:\baseia_v3\corpus\10.1080_096031096334006.pdf
8,Chain-optimazation-pscc-2002.pdf,7.0,0.1597,D:\baseia_v3\corpus\Chain-optimazation-pscc-20...
9,GET17.pdf,8.0,0.1763,D:\baseia_v3\corpus\GET17.pdf


In [5]:
def safe_directory_name(row: pd.Series) -> str:
    document_id = str(row.get("document_id") or "").strip()
    sha256 = str(row.get("sha256") or "").strip()

    if document_id and document_id.lower() != "nan":
        return document_id

    if sha256 and sha256.lower() != "nan":
        return sha256[:16]

    return Path(str(row["path"])).stem

In [6]:
def run_mineru(
    pdf_path: Path,
    output_dir: Path,
    log_path: Path,
    api_url: str
) -> dict[str, object]:
    command = [
        str(MINERU_EXE),
        "-p",
        str(pdf_path),
        "-o",
        str(output_dir),
        "--api-url",
        api_url,
        "--backend",
        BACKEND,
    ]

    started_at = datetime.now(timezone.utc)
    started_counter = time.perf_counter()

    output_dir.mkdir(parents=True, exist_ok=True)
    log_path.parent.mkdir(parents=True, exist_ok=True)

    # print("Comando:")
    # print(subprocess.list2cmdline(command))
    # print()

    with log_path.open(
        "w",
        encoding="utf-8",
        errors="replace",
    ) as log_file:
        process = subprocess.Popen(
            command,
            cwd=PROJECT_ROOT,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding="utf-8",
            errors="replace",
            bufsize=1,
        )

        assert process.stdout is not None

        # for line in process.stdout:
            # print(line, end="")
            # log_file.write(line)
            # log_file.flush()

        return_code = process.wait()

    duration_seconds = time.perf_counter() - started_counter
    completed_at = datetime.now(timezone.utc)

    generated_files = sorted(
        path
        for path in output_dir.rglob("*")
        if path.is_file()
    )

    return {
        "status": "ok" if return_code == 0 else "error",
        "return_code": return_code,
        "started_at": started_at.isoformat(),
        "completed_at": completed_at.isoformat(),
        "duration_seconds": round(duration_seconds, 3),
        "output_dir": str(output_dir.resolve()),
        "log_path": str(log_path.resolve()),
        "generated_files": len(generated_files),
    }

In [7]:
def process_document(
    index: int,
    row: pd.Series,
) -> dict[str, object]:
    pdf_path = Path(
        str(row["path"])
    ).expanduser().resolve()

    document_key = safe_directory_name(row)

    document_output = (
        OUTPUT_ROOT
        / document_key
    )

    log_path = (
        document_output
        / "mineru.log"
    )

    api_url = API_URLS[
        index % len(API_URLS)
    ]

    print(
        f"[{index + 1}/{len(smoke_sample)}] "
        f"{pdf_path.name} → {api_url}"
    )

    # print()
    # print("=" * 100)
    # print(
    #     f"[{index + 1}/{len(smoke_sample)}] "
    #     f"{pdf_path.name}"
    # )
    # print("=" * 100)

    if not pdf_path.exists():
        run = {
            "status": "error",
            "return_code": None,
            "started_at": None,
            "completed_at": None,
            "duration_seconds": None,
            "output_dir": str(
                document_output.resolve()
            ),
            "log_path": str(
                log_path.resolve()
            ),
            "generated_files": 0,
            "error": (
                "Arquivo PDF não encontrado"
            ),
        }

    elif (
        document_output.exists()
        and any(
            document_output.rglob("*.md")
        )
        and not OVERWRITE
    ):
        generated_files = [
            path
            for path
            in document_output.rglob("*")
            if path.is_file()
        ]

        run = {
            "status": "skipped",
            "return_code": 0,
            "started_at": None,
            "completed_at": None,
            "duration_seconds": None,
            "output_dir": str(
                document_output.resolve()
            ),
            "log_path": str(
                log_path.resolve()
            ),
            "generated_files": len(
                generated_files
            ),
            "error": None,
        }

        # print(
        #     f"[{pdf_path.name}] "
        #     "Saída existente; documento ignorado."
        # )

    else:
        try:
            run = run_mineru(
                pdf_path,
                document_output,
                log_path,
                api_url
            )

            run["error"] = None

        except Exception as error:
            run = {
                "status": "error",
                "return_code": None,
                "started_at": None,
                "completed_at": datetime.now(
                    timezone.utc
                ).isoformat(),
                "duration_seconds": None,
                "output_dir": str(
                    document_output.resolve()
                ),
                "log_path": str(
                    log_path.resolve()
                ),
                "generated_files": 0,
                "error": (
                    f"{type(error).__name__}: "
                    f"{error}"
                ),
            }

    page_count = pd.to_numeric(
        pd.Series(
            [
                row.get(
                    "page_count"
                )
            ]
        ),
        errors="coerce",
    ).iloc[0]

    duration = run.get(
        "duration_seconds"
    )

    seconds_per_page = None

    if (
        duration is not None
        and pd.notna(page_count)
        and page_count > 0
    ):
        seconds_per_page = round(
            float(duration)
            / float(page_count),
            4,
        )

    return {
        "document_id": row.get(
            "document_id"
        ),
        "sha256": row.get(
            "sha256"
        ),
        "filename": row.get(
            "filename"
        ),
        "path": str(pdf_path),
        "page_count": page_count,
        "size_mb": row.get(
            "size_mb"
        ),
        "api_url": api_url,
        "backend": BACKEND,
        **run,
        "seconds_per_page": (
            seconds_per_page
        ),
    }

In [8]:
runs: list[dict[str, object]] = []
print(100 * "=")
print(f"Workers: {MAX_WORKERS}")
started_counter = time.perf_counter()

for MAX_WORKERS in [3, 6, 12, 24]:
    
    with ThreadPoolExecutor(
        max_workers=MAX_WORKERS
    ) as executor:
        futures = {
            executor.submit(
                process_document,
                index,
                row.copy(),
            ): index
            for index, row
            in smoke_sample.iterrows()
        }
    
        for completed_count, future in enumerate(
            as_completed(futures),
            start=1,
        ):
            index = futures[future]
    
            try:
                result = future.result()
    
            except Exception as error:
                result = {
                    "document_id": None,
                    "sha256": None,
                    "filename": None,
                    "path": None,
                    "page_count": None,
                    "size_mb": None,
                    "api_url": api_url,
                    "backend": BACKEND,
                    "status": "error",
                    "return_code": None,
                    "started_at": None,
                    "completed_at": datetime.now(
                        timezone.utc
                    ).isoformat(),
                    "duration_seconds": None,
                    "output_dir": None,
                    "log_path": None,
                    "generated_files": 0,
                    "error": (
                        f"Falha no worker do índice "
                        f"{index}: "
                        f"{type(error).__name__}: "
                        f"{error}"
                    ),
                    "seconds_per_page": None,
                }
    
            runs.append(result)
    
            runs_df = pd.DataFrame(
                runs
            )
    
            runs_df.to_parquet(
                RUNS_PATH,
                index=False,
            )
    
            runs_df.to_csv(
                RUNS_CSV_PATH,
                index=False,
                encoding="utf-8-sig",
            )
    
            # print(
            #     f"\nConcluídos: "
            #     f"{completed_count}/"
            #     f"{len(smoke_sample)}"
            # )
    
    
    elapsed_seconds = (
        time.perf_counter()
        - started_counter
    )
    
    completed_df = runs_df[
        runs_df["status"] == "ok"
    ].copy()
    
    total_pages = pd.to_numeric(
        completed_df["page_count"],
        errors="coerce",
    ).sum()
    
    pages_per_minute = (
        total_pages
        / started_counter
        * 60
    )
    

    print(f"Tempo total: {started_counter:.2f} s")
    print(f"Páginas processadas: {total_pages:.0f}")
    print(f"Throughput: {pages_per_minute:.2f} páginas/min")
    print(
        "Erros:",
        int(
            (
                runs_df["status"] == "error"
            ).sum()
        ),
    )
    print(100 * "=")

Workers: 24
[1/24] metadata_williamson_1999_bureaucracies.pdf → https://yl2qubpbnye7sd-8000.proxy.runpod.net
[2/24] Risk-Sharing Contracts and risk management of bilateral contracting in electricity markets [59cd38ef].pdf → https://m05ohopxxf0mkl-8000.proxy.runpod.net
[3/24] New approach to financial time series forecasting - Quantum minimization regularizing BWGC and NGARCH composite model [b94dcebb].pdf → https://r4srxz2kdyhnx9-8000.proxy.runpod.net
[4/24] uerj_rede_sirius_procedimentos_bdtd.pdf → https://yl2qubpbnye7sd-8000.proxy.runpod.net
[5/24] GTL_0081.pdf → https://m05ohopxxf0mkl-8000.proxy.runpod.net
[6/24] GTL_1171.pdf → https://r4srxz2kdyhnx9-8000.proxy.runpod.net
[7/24] GAE_0970.pdf → https://yl2qubpbnye7sd-8000.proxy.runpod.net
[8/24] 10.1080_096031096334006.pdf → https://m05ohopxxf0mkl-8000.proxy.runpod.net
[9/24] Chain-optimazation-pscc-2002.pdf → https://r4srxz2kdyhnx9-8000.proxy.runpod.net
[10/24] GET17.pdf → https://yl2qubpbnye7sd-8000.proxy.runpod.net
[11/24] GMI23.p

In [9]:
runs_df = pd.DataFrame(runs)

runs_df[
    [
        "filename",
        "status",
        "page_count",
        "duration_seconds",
        "seconds_per_page",
        "generated_files",
        "error",
    ]
]

,filename,status,page_count,duration_seconds,seconds_per_page,generated_files,error
0,metadata_williamson_1999_bureaucracies.pdf,error,1.0,18.686,18.6860,9,None
1,New approach to financial time series forecast...,error,4.0,19.959,4.9897,42,None
2,Risk-Sharing Contracts and risk management of ...,error,1.0,20.010,20.0100,10,None
3,uerj_rede_sirius_procedimentos_bdtd.pdf,error,3.0,18.208,6.0693,9,None
4,GTL_1171.pdf,error,5.0,19.337,3.8674,12,None
...,...,...,...,...,...,...,...
91,Risk-Sharing Contracts and risk management of ...,error,1.0,41.911,41.9110,10,None
92,GDS_0748.pdf,error,9.0,41.910,4.6567,32,None
93,10.1080_096031096334006.pdf,error,8.0,42.390,5.2988,24,None
94,GDS08.pdf,error,8.0,42.724,5.3405,29,None


In [10]:
runs_df["status"].value_counts(dropna=False)

status
error    96
Name: count, dtype: int64

In [11]:
successful_runs = runs_df[
    runs_df["status"].isin(["ok", "skipped"])
].copy()

successful_runs[
    [
        "filename",
        "page_count",
        "duration_seconds",
        "seconds_per_page",
        "output_dir",
    ]
].sort_values(
    "seconds_per_page",
    na_position="last",
)

,filename,page_count,duration_seconds,seconds_per_page,output_dir


In [12]:
generated_artifacts = []

for _, row in successful_runs.iterrows():
    output_dir = Path(row["output_dir"])

    for path in output_dir.rglob("*"):
        if not path.is_file():
            continue

        generated_artifacts.append(
            {
                "document_id": row["document_id"],
                "filename": row["filename"],
                "artifact_name": path.name,
                "extension": path.suffix.lower(),
                "size_bytes": path.stat().st_size,
                "path": str(path.resolve()),
            }
        )

artifacts_df = pd.DataFrame(generated_artifacts)

artifacts_df

""


In [13]:
if not artifacts_df.empty:
    artifacts_summary = (
        artifacts_df.groupby(
            [
                "extension",
                "artifact_name",
            ],
            dropna=False,
        )
        .agg(
            quantidade=("path", "size"),
            tamanho_total_mb=(
                "size_bytes",
                lambda values: round(
                    values.sum() / (1024 * 1024),
                    3,
                ),
            ),
        )
        .sort_values(
            "quantidade",
            ascending=False,
        )
        .reset_index()
    )

    display(artifacts_summary)

In [14]:
print(RUNS_PATH.resolve())
print(RUNS_CSV_PATH.resolve())

D:\baseia_v3\artifacts\mineru\smoke\runs.parquet
D:\baseia_v3\artifacts\mineru\smoke\runs.csv
